In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.stats.anova as anova
from statsmodels.stats.outliers_influence import summary_table
from scipy import stats
import matplotlib.pyplot as plt

In [45]:
print("="*78)
print("CODING QUESTION 4: KelleyBlueBookData Dataset Analysis")
print("="*78)

CODING QUESTION 4: KelleyBlueBookData Dataset Analysis


In [3]:
data = pd.read_csv('/Users/lokeshmuvva/Documents/F25_1/regression_f25/data/KelleyBlueBookData.csv')
print(data.head())

         Price  Mileage   Make    Model      Trim   Type  Cylinder  Liter  \
0  17314.10313     8221  Buick  Century  Sedan 4D  Sedan         6    3.1   
1  17542.03608     9135  Buick  Century  Sedan 4D  Sedan         6    3.1   
2  16218.84786    13196  Buick  Century  Sedan 4D  Sedan         6    3.1   
3  16336.91314    16342  Buick  Century  Sedan 4D  Sedan         6    3.1   
4  16339.17032    19832  Buick  Century  Sedan 4D  Sedan         6    3.1   

   Doors  Cruise  Sound  Leather  
0      4       1      1        1  
1      4       1      1        0  
2      4       1      1        0  
3      4       1      0        0  
4      4       1      0        1  


In [4]:
X = data[['Mileage', 'Liter', 'Cylinder']]
y = data['Price']

X = sm.add_constant(X)

In [5]:
# Fit the model
model = smf.ols('Price ~ Mileage + Liter + Cylinder',data=data).fit()

In [6]:
seq = sm.stats.anova_lm(model, typ=1)
seq

,df,sum_sq,mean_sq,F,PR(>F)
Mileage,1.0,1.605590e+09,1.605590e+09,24.890171,7.448191e-07
Liter,1.0,2.421824e+10,2.421824e+10,375.435819,7.134623e-69
Cylinder,1.0,1.031948e+09,1.031948e+09,15.997457,6.930502e-05
Residual,800.0,5.160560e+10,6.450701e+07,NaN,NaN


In [58]:
print("="*50)
print("PART (a): Model Interpretation")
print("="*50)

# Extract F and p-value for Cylinder
f_val = seq.loc["Cylinder", "F"]
p_val = seq.loc["Cylinder", "PR(>F)"]

print(f"\nF value for Cylinder: {f_val:.4f}, p-value: {p_val:.4g}")

print("\nInterpretation:")
print("H₀: β_Cylinder = 0 (Cylinder adds no predictive value beyond Mileage and Liter)")
print("H₁: β_Cylinder ≠ 0 (Cylinder adds significant predictive value beyond Mileage and Liter)")
if p_val < 0.05:
    print(f"P-value = {p_val:.6f} < 0.05, so we reject H₀")
    print("Conclusion: Adding Cylinder significantly improves the model that already contains Mileage and Liter")
else:
    print(f"P-value = {p_val:.6f} ≥ 0.05, so we fail to reject H₀")
    print("Conclusion: Adding Cylinder does not significantly improve the model that already contains Mileage and Liter")

PART (a): Model Interpretation

F value for Cylinder: 15.9975, p-value: 6.931e-05

Interpretation:
H₀: β_Cylinder = 0 (Cylinder adds no predictive value beyond Mileage and Liter)
H₁: β_Cylinder ≠ 0 (Cylinder adds significant predictive value beyond Mileage and Liter)
P-value = 0.000069 < 0.05, so we reject H₀
Conclusion: Adding Cylinder significantly improves the model that already contains Mileage and Liter


In [57]:
print("="*50)
print("PART (b): Manual ANOVA Test")
print("="*50)

null_model = smf.ols('Price ~ Mileage + Liter',data=data).fit()
full_model = smf.ols('Price ~ Mileage + Liter + Cylinder',data=data).fit()

seq_null = sm.stats.anova_lm(null_model, typ=1)
seq_full = sm.stats.anova_lm(full_model, typ=1)

print("Null Model")
print(seq_null)
print("\nFull Model")
print(seq_full)

SSE_H0 = seq_null.loc['Residual', 'sum_sq']
df_H0 = seq_null.loc['Residual', 'df']

SSE_H1 = seq_full.loc['Residual', 'sum_sq']
df_H1 = seq_full.loc['Residual', 'df']

print(f"\nSSE_H0 = {SSE_H0}")
print(f"df_H0 = {df_H0}")
print(f"SSE_H1 = {SSE_H1}")
print(f"df_H1 = {df_H1}")

numerator = (SSE_H0 - SSE_H1) /(df_H0 - df_H1)
denominator = (SSE_H1/df_H1)

print(f"\nNumerator = ({SSE_H0} - {SSE_H1})/({df_H0} - {df_H1})")
print(f"Denominator = ({SSE_H1}/{df_H1})")

manual_F_stat = numerator/denominator

print(f"\nThe manually calculated F statistic is {manual_F_stat:.4f}")
print(f"The F-value from the original table is {f_val:.4f}")
if np.isclose(manual_F_stat, f_val):
    print("\nThis matches the F-statistic calculated from part (a)")
else:
    print("\nThis does not match the F-statistic calculated from part (a)")

PART (b): Manual ANOVA Test
Null Model
             df        sum_sq       mean_sq           F        PR(>F)
Mileage     1.0  1.605590e+09  1.605590e+09   24.432707  9.374438e-07
Liter       1.0  2.421824e+10  2.421824e+10  368.535574  7.328735e-68
Residual  801.0  5.263755e+10  6.571480e+07         NaN           NaN

Full Model
             df        sum_sq       mean_sq           F        PR(>F)
Mileage     1.0  1.605590e+09  1.605590e+09   24.890171  7.448191e-07
Liter       1.0  2.421824e+10  2.421824e+10  375.435819  7.134623e-69
Cylinder    1.0  1.031948e+09  1.031948e+09   15.997457  6.930502e-05
Residual  800.0  5.160560e+10  6.450701e+07         NaN           NaN

SSE_H0 = 52637552166.235855
df_H0 = 801.0
SSE_H1 = 51605604120.22623
df_H1 = 800.0

Numerator = (52637552166.235855 - 51605604120.22623)/(801.0 - 800.0)
Denominator = (51605604120.22623/800.0)

The manually calculated F statistic is 15.9975
The F-value from the original table is 15.9975

This matches the F-statistic 

In [56]:
print("="*50)
print("PART (c): Partial (Type II) Sum of Squares")
print("="*50)

# Type II (Partial) ANOVA
anova_type2 = anova.anova_lm(model, typ=2)
print("Type II (Partial) ANOVA:")
print(anova_type2)

cylinder_f_type2 = anova_type2.loc['Cylinder', 'F']
cylinder_p_type2 = anova_type2.loc['Cylinder', 'PR(>F)']

print(f"\nF-value for Cylinder from partial ANOVA (typ=2): {cylinder_f_type2:.6f}")
print(f"P-value for Cylinder from partial ANOVA (typ=2): {cylinder_p_type2:.6f}")

print(f"\nF-value for Cylinder from part (a): {f_val:.4f}, p-value: {p_val:.4g}")

if np.isclose(cylinder_f_type2, f_val) and np.isclose(cylinder_p_type2, p_val):
    print("\nThis matches the F-test calculated from part (a)")
else:
    print("\nThis does not match the F-test calculated from part (a)")

PART (c): Partial (Type II) Sum of Squares
Type II (Partial) ANOVA:
                sum_sq     df          F    PR(>F)
Mileage   1.283997e+09    1.0  19.904763  0.000009
Liter     1.929760e+08    1.0   2.991552  0.084086
Cylinder  1.031948e+09    1.0  15.997457  0.000069
Residual  5.160560e+10  800.0        NaN       NaN

F-value for Cylinder from partial ANOVA (typ=2): 15.997457
P-value for Cylinder from partial ANOVA (typ=2): 0.000069

F-value for Cylinder from part (a): 15.9975, p-value: 6.931e-05

This matches the F-test calculated from part (a)


The F-stat for Cylinder is the same in both cases because of the order in which both types are arranged when deriving these values. In typ = 1, Cylinder is last in the order, so the full model adds Cylinder after already including Mileage and Liter (in that order). In typ = 2, the full model adds Cylinder after controlling for the other predictors. The order in which Cylinder is added is the same in both types (added at the very end) which is also why the F-statistic is the same. 

In [55]:
print("="*50)
print("PART (d): Partial (Type II) Sum of Squares")
print("="*50)

print("\nInterpretation of 'Mileage' in Type II ANOVA:")
print("- Reduced model: Price ~ Liter + Cylinder")
print("- Full model: Price ~ Mileage + Liter + Cylinder")
print("- Partial SS tests: Is 'Mileage' needed given that 'Liter' and 'Cylinder' are in the model?")
print("- H₀: β_Mileage = 0 (given Liter and Cylinder are in the model)")
print("- H₁: β_Mileage ≠ 0 (given Liter and Cylinder are in the model)")

mileage_f_type2 = anova_type2.loc['Mileage', 'F']
mileage_p_type2 = anova_type2.loc['Mileage', 'PR(>F)']

print(f"\nANOVA (typ=2) F-value for Mileage: {mileage_f_type2:.4f}, p-value: {mileage_p_type2:.4g}")

if mileage_p_type2 < 0.05:
    print(f"P-value = {mileage_p_type2:.6f} < 0.05, so we reject H₀")
    print("Conclusion: Adding Mileage significantly improves the model that already contains Liter and Cylinder")
else:
    print(f"P-value = {mileage_p_type2:.6f} ≥ 0.05, so we fail to reject H₀")
    print("Conclusion: Adding Mileage does not significantly improve the model that already contains Liter and Cylinder")

PART (d): Partial (Type II) Sum of Squares

Interpretation of 'Mileage' in Type II ANOVA:
- Reduced model: Price ~ Liter + Cylinder
- Full model: Price ~ Mileage + Liter + Cylinder
- Partial SS tests: Is 'Mileage' needed given that 'Liter' and 'Cylinder' are in the model?
- H₀: β_Mileage = 0 (given Liter and Cylinder are in the model)
- H₁: β_Mileage ≠ 0 (given Liter and Cylinder are in the model)

ANOVA (typ=2) F-value for Mileage: 19.9048, p-value: 9.306e-06
P-value = 0.000009 < 0.05, so we reject H₀
Conclusion: Adding Mileage significantly improves the model that already contains Liter and Cylinder


In [54]:
print("="*50)
print("PART (e): Manual ANOVA (typ=2) Test")
print("="*50)

null_model_e = smf.ols('Price ~ Liter + Cylinder',data=data).fit()
full_model_e = smf.ols('Price ~ Mileage + Liter + Cylinder',data=data).fit()

par_null = sm.stats.anova_lm(null_model_e, typ=2)
par_full = sm.stats.anova_lm(full_model_e, typ=2)

print("Null Model")
print(par_null)
print("\nFull Model")
print(par_full)

SSE_H0_e = par_null.loc['Residual', 'sum_sq']
df_H0_e = par_null.loc['Residual', 'df']

SSE_H1_e = par_full.loc['Residual', 'sum_sq']
df_H1_e = par_full.loc['Residual', 'df']

numerator_e = (SSE_H0_e - SSE_H1_e) /(df_H0_e - df_H1_e)
denominator_e = (SSE_H1_e/df_H1_e)

print(f"\nNumerator = ({SSE_H0_e} - {SSE_H1_e})/({df_H0_e} - {df_H1_e})")
print(f"Denominator = ({SSE_H1_e}/{df_H1_e})")

manual_F_stat_e = numerator_e/denominator_e

print(f"\nThe manually calculated F statistic is {manual_F_stat_e:.4f}")
print(f"\nANOVA (typ=2) F-value for Mileage from part (d): {mileage_f_type2:.4f}")
if np.isclose(manual_F_stat_e, mileage_f_type2):
    print("\nThis matches the F-statistic calculated from part (d)")
else:
    print("\nThis does not match the F-statistic calculated from part (d)")

PART (e): Manual ANOVA (typ=2) Test
Null Model
                sum_sq     df          F    PR(>F)
Liter     1.613541e+08    1.0   2.443669  0.118395
Cylinder  1.128963e+09    1.0  17.097866  0.000039
Residual  5.288960e+10  801.0        NaN       NaN

Full Model
                sum_sq     df          F    PR(>F)
Mileage   1.283997e+09    1.0  19.904763  0.000009
Liter     1.929760e+08    1.0   2.991552  0.084086
Cylinder  1.031948e+09    1.0  15.997457  0.000069
Residual  5.160560e+10  800.0        NaN       NaN

Numerator = (52889600780.65452 - 51605604120.22623)/(801.0 - 800.0)
Denominator = (51605604120.22623/800.0)

The manually calculated F statistic is 19.9048

ANOVA (typ=2) F-value for Mileage from part (d): 19.9048

This matches the F-statistic calculated from part (d)


In [39]:
print("\n" + "="*78)
print("CODING QUESTION 5: KelleyBlueBookData Dataset Analysis")
print("="*78)


CODING QUESTION 5: KelleyBlueBookData Dataset Analysis


In [71]:
print("="*50)
print("PART (a): Model Fitting and Summary")
print("="*50)

# Fit the model
model_5 = smf.ols('Price ~ Mileage + Cylinder',data=data).fit()
print(model_5.summary())
print(f"\nThe R2 value from a model with Mileage and Cylinder as predictors is {model_5.rsquared}")
print(f"The Adjusted R2 value is {model_5.rsquared_adj}")

PART (a): Model Fitting and Summary
                            OLS Regression Results                            
Dep. Variable:                  Price   R-squared:                       0.340
Model:                            OLS   Adj. R-squared:                  0.338
Method:                 Least Squares   F-statistic:                     206.2
Date:                Sun, 28 Sep 2025   Prob (F-statistic):           5.95e-73
Time:                        18:06:51   Log-Likelihood:                -8369.2
No. Observations:                 804   AIC:                         1.674e+04
Df Residuals:                     801   BIC:                         1.676e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   3145

In [63]:
print("="*50)
print("PART (b): Manual R2 and Adjusted R2 Calculations")
print("="*50)

SSE = model_5.ssr  # or model_5.scale * model_5.df_resid
SST = model_5.centered_tss  # Total sum of squares
n = model_5.nobs  # Number of observations
p = model_5.df_model + 1  # Parameters (predictors + intercept)

print(f"\nSSE = {SSE}")
print(f"SST = {SST}")
print(f"n = {n}")
print(f"p = {p}")

r_squared = 1 - (SSE/SST)
r_squared_adj = 1 - ((SSE/(n-p))/(SST/(n-1)))

print(f"\nR2 = 1 - ({SSE}/{SST})")
print(f"R2 = {r_squared}")
print(f"\nAdjusted R2  = 1 - (({SSE}/({n}-{p}))/({SST}({n}-1)))")
print(f"Adjusted R2 = {r_squared_adj}")

if np.isclose(r_squared, model_5.rsquared) and np.isclose(r_squared_adj, model_5.rsquared_adj):
    print("\nThese values match with the python outputs in (a) above")
else:
    print("\nThese values do not match with the python outputs in (a) above")

PART (b): Manual R2 and Adjusted R2 Calculations

SSE = 51798580167.8953
SST = 78461382864.00787
n = 804.0
p = 3.0

R2 = 1 - (51798580167.8953/78461382864.00787)
R2 = 0.33982070826263044

Adjusted R2  = 1 - ((51798580167.8953/(804.0-3.0))/(78461382864.00787(804.0-1)))
Adjusted R2 = 0.33817232051796786

These values match with the python outputs in (a) above


In [72]:
print("="*50)
print("PART (c): Model Fitting and Summary")
print("="*50)

# Fit the model
model_5c = smf.ols('Price ~ Mileage + Liter + Cylinder',data=data).fit()
print(model_5c.summary())
print(f"The R2 value from a model with Mileage, Liter and Cylinder as predictors is {model_5c.rsquared}")
print(f"The Adjusted R2 value is {model_5c.rsquared_adj}")
print(f"The Adjusted R2 value increased by approximately {model_5c.rsquared_adj - model_5.rsquared_adj}")

PART (c): Model Fitting and Summary
                            OLS Regression Results                            
Dep. Variable:                  Price   R-squared:                       0.342
Model:                            OLS   Adj. R-squared:                  0.340
Method:                 Least Squares   F-statistic:                     138.8
Date:                Sun, 28 Sep 2025   Prob (F-statistic):           2.18e-72
Time:                        18:07:17   Log-Likelihood:                -8367.7
No. Observations:                 804   AIC:                         1.674e+04
Df Residuals:                     800   BIC:                         1.676e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   4707

R squared did increase when Liter was added as a predictor. However, this always happens as adding more predictors can never reduce the R squared value, it will always try to fit according to all the predictors added. The adjusted R squared value also increased by approximately 0.0016414420630816995. This means that Model 2 is more preferable according to the adjusted R squared value, indicating that Liter as a predictor is worth including adding. The adjusted R squared value penalizes for the addition of parameters. The fact that adding another predictor in Liter increased the adjusted R squared value means that there is a significant contribution from Liter as a predictor.

In [70]:
print("="*50)
print("PART (d): Open question")
print("="*50)

PART (d): Open question


From the type 2 ANOVA in question 4 part (c), the p-value was 0.084086 which is greater than 0.05, indicating that it is not significant. The adjusted R squared improvement is also not practically meaningful as the difference is about 0.16%. The t-stat from the OLS regression result is also very small (1.730), indicating that Liter's contribution is marginal given Mileage and Cylinder are already in the model. If Liter isn't statistically significant when controlling for Cylinder, this means that there is a multicollinear relationship between them. It doesn't make sense to choose a more complex model for a 0.16% improvement in the adjusted R squared value, when the added predictor is not even significant From a practical standpoint, both liter and cylinders measure the size of the engine so adding liter does not add meaningful pricing information. Therefore choosing Model 1 makes the most sense in terms of the statistics and real-life meaning and applicability. 